# 00b. NumPy 80/20 Mastery Guide (Tensors, Shapes & Vectorized Math)

Deep Learning is **90% matrix multiplication and shape manipulation**.  
NumPy (`np`) is the foundation of Python scientific computing—and **PyTorch and TensorFlow copy NumPy's exact API and broadcasting rules**. Once NumPy clicks, PyTorch & TensorFlow become 10x easier.

### What You Will Master (80/20 Rule):
1. Creating arrays (`np.array`, `zeros`, `ones`, `randn`)
2. Inspecting `.shape`, `.ndim`, `.dtype`
3. Slicing & Boolean Masks
4. Element-wise (`*`) vs Matrix Multiplication (`@`)
5. `axis=0` vs `axis=1`
6. **Broadcasting** (and the #1 silent shape bug!)
7. Reshaping, `np.newaxis`, `.transpose()`
8. Real Neural Net math in NumPy (Sigmoid, Softmax, Cross-Entropy)

## 1. Creating NumPy Arrays (Vectors, Matrices & Batches)

Remember: a Python `list` (`[1, 2, 3]`) cannot do vector math. Wrapping it in `np.array(...)` converts it into a fast C-backed `ndarray`.

### 4 Essential Constructors:
* `np.array([...])` — convert a Python list into an array
* `np.zeros((rows, cols))` — all zeros (used for initial bias vectors `b`!)
* `np.ones((rows, cols))` — all ones
* `np.random.randn(rows, cols)` — Gaussian random numbers (mean `0`, std `1`), used to initialize weight matrices `W`!

In [ ]:
import numpy as np

np.random.seed(42)  # makes random numbers reproducible every run

vec = np.array([1.0, 2.0, 3.0])          # 1D vector
mat = np.array([[1, 2, 3], [4, 5, 6]])   # 2D matrix (2 rows, 3 cols)
W = np.random.randn(3, 4) * 0.01         # Weight matrix (3 inputs -> 4 outputs)
b = np.zeros((1, 4))                     # Bias vector

print("Vector:", vec)
print("Matrix:\n", mat)
print("Small random W:\n", W)

## 2. Inspecting Shapes (`.shape`, `.ndim`, `.dtype`)

**Rule #1 of Debugging ML:** Whenever anything breaks, print `.shape` and `.dtype` first!

### Standard Deep Learning Shape Conventions:
* **1D:** `(N,)` — a flat list of `N` numbers (e.g., labels `y`).
* **2D (Tabular / MLP):** `(Batch, Features)` — e.g. `(32, 128)` means 32 patients, 128 genes each.
* **3D (RNN / Sequence / EEG):** `(Batch, TimeSteps, Features)` — e.g. `(32, 100, 64)`.
* **4D (CNN / Spectrogram / Image):** `(Batch, Channels, Height, Width)`.

> **Float vs Int:** Weights & inputs are usually `float32` / `float64`; class labels (`0, 1, 2`) and token IDs are `int64`.

In [ ]:
x_tabular = np.zeros((32, 128), dtype=np.float32)   # 32 samples, 128 features
x_sequence = np.zeros((32, 50, 16))                 # 32 sequences, length 50, dim 16
y_labels = np.array([0, 1, 1, 0])                   # 1D integer vector

print("Tabular shape:", x_tabular.shape, "| ndim:", x_tabular.ndim, "| dtype:", x_tabular.dtype)
print("Sequence shape:", x_sequence.shape, "| ndim:", x_sequence.ndim)
print("Labels shape:", y_labels.shape, "| dtype:", y_labels.dtype)

## 3. Indexing, Slicing & Boolean Masking

Grabbing parts of a 2D matrix uses `[row_slice, col_slice]`:
* `X[0, :]` (or `X[0]`) → entire **row 0**
* `X[:, 0]` → entire **column 0** (`:` means "keep everything along this axis")
* `X[:, -1]` → **last** column (in a 3D RNN array `(B, T, H)`, `h[:, -1, :]` grabs the **last timestep**!)
* **Boolean masking (`X[mask]`):** `vals > 0` creates a `True/False` array; passing it inside `[...]` filters matching elements **without a `for` loop**!

In [ ]:
X = np.array([
    [10, 20, 30],
    [40, 50, 60],
    [70, 80, 90],
])

print("Row 1:", X[1, :])
print("Column 2 (last col):", X[:, -1])
print("Top-left 2x2 block:\n", X[:2, :2])

# Boolean masking (used in ReLU and filtering positive samples)
scores = np.array([-1.5, 0.2, 2.8, -0.1, 4.0])
mask = scores > 0
print("Boolean mask:", mask)
print("Positive scores only:", scores[mask])

# Setting negative values to 0 in place (how ReLU works internally!):
relu_out = scores.copy()
relu_out[relu_out < 0] = 0
print("ReLU result:", relu_out)

## 4. Element-Wise (`*`) vs. Matrix Multiplication (`@`)

**Never confuse these two!**
* `A * B`: **Element-wise** multiplication (multiplies matching positions). Shapes must match (or broadcast).
* `A @ B`: **Matrix multiplication (dot product)** — the core equation of every neural network layer `Y = X @ W + b`.
  * Rule for `@`: inner dimensions MUST match! `(B, D_in) @ (D_in, D_out) -> (B, D_out)`.

In [ ]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[10, 20], [30, 40]])

print("Element-wise A * B:\n", A * B)
print("Matrix multiply A @ B:\n", A @ B)

# Neural layer shape check: (Batch=5, In=4) @ (In=4, Out=3) -> (5, 3)
X_batch = np.ones((5, 4))
W = np.random.randn(4, 3)
b = np.zeros((1, 3))
Y = X_batch @ W + b
print("Layer output shape:", Y.shape)

## 5. The `axis` Rule (`axis=0` vs `axis=1`)

In operations like `.sum()`, `.mean()`, `.max()`, `.argmax()`:
**Golden Rule:** **`axis=k` collapses (squashes) dimension `k`!**

Suppose `X` has shape `(3, 4)` → `(3 samples, 4 features)`:
* `X.mean(axis=0)` → collapses axis 0 (the 3 samples) → shape `(4,)` = **average of each feature**!
* `X.mean(axis=1)` → collapses axis 1 (the 4 features) → shape `(3,)` = **average of each sample**!
* `keepdims=True` keeps the collapsed axis as size `1` (e.g. `(3, 1)` instead of `(3,)`), which is **critical** for safe broadcasting!

In [ ]:
# 3 patients (rows) x 4 genes (columns)
X = np.array([
    [1.0, 2.0, 3.0, 4.0],
    [5.0, 6.0, 7.0, 8.0],
    [9.0, 10.0, 11.0, 12.0],
])

print("Mean per gene (axis=0):", X.mean(axis=0), "shape:", X.mean(axis=0).shape)
print("Mean per patient (axis=1):", X.mean(axis=1), "shape:", X.mean(axis=1).shape)
print("Mean per patient with keepdims=True:\n", X.mean(axis=1, keepdims=True),
      "shape:", X.mean(axis=1, keepdims=True).shape)

# argmax(axis=1): convert class probabilities (3 samples x 4 classes) to predicted class ID!
probs = np.array([[0.1, 0.7, 0.1, 0.1],
                  [0.8, 0.1, 0.05, 0.05],
                  [0.1, 0.2, 0.2, 0.5]])
preds = probs.argmax(axis=1)
print("Predicted classes:", preds)  # [1, 0, 3]

## 6. Broadcasting (and the #1 Silent Bug!)

**Broadcasting** lets NumPy automatically stretch an array of size `1` across a larger dimension during `+`, `-`, `*`, `/`.

### How Broadcasting Works (Compare Shapes Right-to-Left):
* `(3, 4)` minus `(1, 4)` → `1` stretches to `3` → works cleanly! Shape stays `(3, 4)`.
* `(3, 4)` minus `(3, 1)` → `1` stretches to `4` → works cleanly! Shape stays `(3, 4)`.

### THE #1 SILENT BUG: `(N,) - (N, 1)`!
If `y_pred` has shape `(3,)` and `y_true` has shape `(3, 1)`, NumPy stretches **both** into a **`(3, 3)` matrix** instead of throwing an error! Always keep `keepdims=True` or check shapes!

In [ ]:
X = np.array([[10.0, 200.0],
              [20.0, 400.0],
              [30.0, 600.0]])  # shape (3, 2)

# Standardizing each feature (column): mean & std have shape (1, 2)
mu = X.mean(axis=0, keepdims=True)     # shape (1, 2)
sigma = X.std(axis=0, keepdims=True)   # shape (1, 2)
X_norm = (X - mu) / sigma              # (3, 2) - (1, 2) -> (3, 2)!
print("Normalized X:\n", X_norm)

# SILENT BUG DEMO: (3,) - (3, 1) accidentally creates (3, 3)!
a = np.array([1, 2, 3])          # shape (3,)
b = np.array([[1], [2], [3]])    # shape (3, 1)
print("BUGGY broadcast shape (a - b):", (a - b).shape)  # (3, 3) instead of (3,)!

## 7. Reshaping, `np.newaxis` (`None`), `.squeeze()`, `.transpose()`

Deep learning models are picky about tensor dimensions:
* `.reshape(new_shape)` rearranges elements without changing them (`-1` tells NumPy *"compute this size automatically"*).
* `arr[:, None]` (or `np.newaxis`) inserts a dimension of size `1` (e.g., turning `(3,)` into `(3, 1)`).
* `.squeeze()` removes size-`1` dimensions (e.g., `(3, 1) -> (3,)`).
* `.transpose(...)` or `.T` swaps axes (vital when converting `(Time, Channels)` ↔ `(Channels, Time)` for CNNs/EEGs!).

In [ ]:
a = np.arange(12)  # [0, 1, ..., 11], shape (12,)

# Reshape into (3, 4) or auto-infer using -1
m = a.reshape(3, -1)  # NumPy figures out -1 must be 4!
print("Reshaped (3, 4):\n", m)

# Add a dimension with None (newaxis) vs squeeze()
y = np.array([0, 1, 1])       # shape (3,)
y_col = y[:, None]            # shape (3, 1)
y_flat = y_col.squeeze()      # back to shape (3,)
print("Shapes:", y.shape, "->", y_col.shape, "->", y_flat.shape)

# Transpose 2D (.T) and swap axes on 3D (Batch, Time, Channels) -> (Batch, Channels, Time)
eeg_btc = np.zeros((8, 250, 64))
eeg_bct = eeg_btc.transpose(0, 2, 1)  # PyTorch Conv1d wants Channels BEFORE Time!
print("EEG transposed:", eeg_btc.shape, "->", eeg_bct.shape)

## 8. Core Neural Net Functions in Pure NumPy (Sigmoid, Softmax, Cross-Entropy)

Every neural network paper uses these exact 3 formulas:
1. **Sigmoid:** $\sigma(z) = \frac{1}{1 + e^{-z}}$ (squashes logits to $(0, 1)$ probability)
2. **Numerically Stable Softmax:** Subtract `max(z)` before `exp(z)` so `exp(1000)` doesn't overflow to `inf`/`NaN`!
3. **Cross-Entropy Loss:** $-\frac{1}{B} \sum \log(\hat{p}_{\text{true class}} + 10^{-12})$

In [ ]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-z))

def softmax(logits: np.ndarray) -> np.ndarray:
    # Subtract max along class axis (axis=-1) for numerical stability!
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exps = np.exp(shifted)
    return exps / np.sum(exps, axis=-1, keepdims=True)

def cross_entropy(probs: np.ndarray, targets: np.ndarray) -> float:
    batch_size = len(targets)
    # Fancy indexing: pick probability of the true class for each row!
    correct_probs = probs[np.arange(batch_size), targets]
    return -np.mean(np.log(correct_probs + 1e-12))

logits = np.array([[2.0, 0.5, -1.0],
                   [0.1, 3.0, 0.2]])
targets = np.array([0, 1])

probs = softmax(logits)
print("Probabilities (each row sums to 1):\n", probs)
print("Row sums:", probs.sum(axis=-1))
print("Cross-entropy loss:", cross_entropy(probs, targets))

## 9. Combining Arrays (`np.concatenate` vs `np.stack`)

* **`np.concatenate([A, B], axis=0)`**: Joins along an **existing** axis (e.g., two batches `(16, 4)` + `(16, 4)` → `(32, 4)`).
* **`np.stack([A, B], axis=0)`**: Creates a **brand new** axis (e.g., stacking 10 spectrograms of shape `(64, 128)` → `(10, 64, 128)`).

In [ ]:
batch1 = np.ones((2, 3))
batch2 = np.zeros((2, 3))

cat_rows = np.concatenate([batch1, batch2], axis=0)   # (4, 3) — stacked vertically
cat_cols = np.concatenate([batch1, batch2], axis=1)   # (2, 6) — glued side by side
stacked = np.stack([batch1, batch2], axis=0)          # (2, 2, 3) — NEW axis 0!

print("concatenate axis=0:", cat_rows.shape)
print("concatenate axis=1:", cat_cols.shape)
print("stack axis=0:", stacked.shape)

---
## Practice Challenges (Try Without Peeking!)

1. Create a `(4, 5)` random matrix `X`. Normalize **each row** (not each column!) to have mean `0` and std `1` using `axis=1, keepdims=True`. Verify `X_norm.mean(axis=1)` is near zero.
2. Given `logits = np.random.randn(6, 3)` and `targets = np.array([0, 2, 1, 1, 0, 2])`, compute predicted classes with `argmax` and calculate classification accuracy in 1 line using `np.mean(preds == targets)`.
3. Simulate combining a tabular vector `x = np.ones((10, 4))` with a `Linear(4 -> 8)` → `ReLU` → `Linear(8 -> 2)` forward pass in NumPy and verify the output shape is `(10, 2)`.